In [5]:
import numpy as np
import pandas as pd
from scipy import stats

# Paramètres réalistes pour l'aviculture
np.random.seed(42)
start = "2018-01-01"
end = "2025-12-31"
freq = "1H"
timestamps = pd.date_range(start=start, end=end, freq='h')
n = len(timestamps)

print("Génération de données avicoles réalistes...")

# === CONFIGURATION DES PARAMÈTRES RÉALISTES ===
params = {
    'temp': {'min': 18, 'max': 28, 'ideal': (20, 24)},
    'humidity': {'min': 40, 'max': 70, 'ideal': (50, 60)},
    'co': {'base': 0.002, 'max': 0.03},
    'lpg': {'base': 0.001, 'max': 0.02},
    'smoke': {'base': 0.001, 'max': 0.015},
    'nh3': {'base': 0.005, 'max': 0.04}  # NH3 plus élevé en élevage
}

# Variables temporelles détaillées
hour = timestamps.hour
day_of_year = timestamps.dayofyear
month = timestamps.month
day_of_week = timestamps.dayofweek
is_weekend = (day_of_week >= 5)
is_night = (hour < 6) | (hour > 21)
is_day = ~is_night
is_working_hours = (hour >= 7) & (hour <= 18)

# === SIGNAL DE BASE AVEC MULTI-SAISONNALITÉ ===
regular = np.arange(n)

# Saisonnalité annuelle (plus chaud en été)
annual_season = 2 * np.sin(day_of_year/365 * 2*np.pi - np.pi/2)

# Saisonnalité hebdomadaire (activité différente weekdays/weekends)
weekly_season = 0.5 * np.sin(day_of_week/7 * 2*np.pi)

# Saisonnalité journalière
daily_season = 3 * np.sin((hour-14)/24 * 2*np.pi)  # Pic à 14h

# === TEMPÉRATURE ULTRA-RÉALISTE ===
base_temp = 21.5 + annual_season  # Base + saison annuelle
temp_variation = daily_season + weekly_season * 0.3

# Ajout d'effets météorologiques aléatoires
weather_effect = np.zeros(n)
for i in range(1, n):
    weather_effect[i] = 0.95 * weather_effect[i-1] + np.random.normal(0, 0.3)
weather_effect = np.clip(weather_effect, -2, 2)

temperature = base_temp + temp_variation + weather_effect + np.random.normal(0, 0.2, n)
temperature = np.clip(temperature, params['temp']['min'], params['temp']['max'])

# === HUMIDITÉ CORRÉLÉE AVEC TEMPÉRATURE ===
base_humidity = 55 - annual_season * 3  # Plus humide en hiver
humidity_variation = -daily_season * 1.5  # Inverse de la température

# Effet de l'humidité résiduelle
humidity_effect = np.zeros(n)
for i in range(1, n):
    humidity_effect[i] = 0.9 * humidity_effect[i-1] + np.random.normal(0, 1)
humidity_effect = np.clip(humidity_effect, -5, 5)

humidity = base_humidity + humidity_variation + humidity_effect + np.random.normal(0, 1, n)
humidity = np.clip(humidity, params['humidity']['min'], params['humidity']['max'])

# === GAZ AVEC COMPORTEMENTS SPÉCIFIQUES ===

# CO - Corrélé avec l'activité et la température
co_base = params['co']['base']
co_activity = 0.01 * (temperature - 20) / 5  # Augmente avec la température
co_peak_prob = 0.001  # Probabilité de pics de CO
co_peaks = np.random.binomial(1, co_peak_prob, n) * np.random.exponential(0.01, n)
co = np.abs(co_base + co_activity + co_peaks + np.random.normal(0, 0.001, n))

# LPG - Stable avec pics aléatoires
lpg_base = params['lpg']['base']
lpg_peaks_prob = 0.0005
lpg_peaks = np.random.binomial(1, lpg_peaks_prob, n) * np.random.exponential(0.005, n)
lpg = np.abs(lpg_base + lpg_peaks + np.random.normal(0, 0.0005, n))

# Smoke - Rare mais possible
smoke_base = params['smoke']['base']
smoke_prob = 0.0002
smoke_events = np.random.binomial(1, smoke_prob, n) * np.random.exponential(0.005, n)
smoke = np.abs(smoke_base + smoke_events + np.random.normal(0, 0.0003, n))

# NH3 - Spécifique à l'élevage, augmente avec l'humidité et le temps
nh3_base = params['nh3']['base']
nh3_humidity_effect = 0.01 * (humidity - 50) / 20  # Augmente avec l'humidité
nh3_buildup = np.zeros(n)
for i in range(1, n):
    nh3_buildup[i] = 0.999 * nh3_buildup[i-1] + np.random.normal(0, 0.0001)
nh3 = np.abs(nh3_base + nh3_humidity_effect + nh3_buildup + np.random.normal(0, 0.002, n))

# Clip tous les gaz à leurs maximums
co = np.clip(co, 0, params['co']['max'])
lpg = np.clip(lpg, 0, params['lpg']['max'])
smoke = np.clip(smoke, 0, params['smoke']['max'])
nh3 = np.clip(nh3, 0, params['nh3']['max'])

# === LUMIÈRE INTELLIGENTE ===
# Base: lumière allumée le jour
light = is_day.astype(int)

# Ajout de variations réalistes:
# - Parfois allumée la nuit (soins d'urgence)
# - Parfois éteinte le jour (économie d'énergie)
night_light_prob = 0.02  # 2% chance lumière allumée la nuit
day_light_off_prob = 0.05  # 5% chance lumière éteinte le jour

night_lights = is_night & (np.random.random(n) < night_light_prob)
day_lights_off = is_day & (np.random.random(n) < day_light_off_prob)

light[night_lights] = 1
light[day_lights_off] = 0

# Pattern spécifique: lumière toujours allumée pendant les soins
morning_care = (hour >= 5) & (hour <= 7)
evening_care = (hour >= 17) & (hour <= 19)
light[morning_care | evening_care] = 1

# === MOUVEMENT INTELLIGENT ===
# Probabilité de base dépendante du temps
base_motion_prob = np.zeros(n)
base_motion_prob[is_night] = 0.05  # 5% la nuit
base_motion_prob[is_day] = 0.15    # 15% le jour
base_motion_prob[is_working_hours] = 0.25  # 25% heures de travail

# Augmentation pendant les soins
care_hours = ((hour >= 5) & (hour <= 8)) | ((hour >= 16) & (hour <= 19))
base_motion_prob[care_hours] = 0.4

# Weekends: activité réduite
base_motion_prob[is_weekend] *= 0.7

# Corrélation avec la lumière
light_effect = light * 0.2
base_motion_prob += light_effect

# Effet de la température (plus d'activité dans la plage idéale)
temp_effect = np.where(
    (temperature >= 20) & (temperature <= 24),
    0.1,  # Plage idéale
    np.where(temperature < 18, -0.05, -0.08)  # Trop froid ou trop chaud
)
base_motion_prob += temp_effect

# Génération du mouvement
motion_prob = np.clip(base_motion_prob, 0.01, 0.8)
motion = np.random.binomial(1, p=motion_prob)

# === ANOMALIES ET ÉVÉNEMENTS RARES - VERSION CORRIGÉE ===
def add_anomalies(data, anomaly_prob=0.001, anomaly_strength=3):
    """Ajoute des anomalies réalistes - version corrigée"""
    # S'assurer que data est un array numpy
    data_array = np.array(data) if not isinstance(data, np.ndarray) else data.copy()
    anomalies = np.random.binomial(1, anomaly_prob, len(data_array))
    anomaly_indices = np.where(anomalies)[0]
    anomaly_values = np.random.normal(0, anomaly_strength, len(anomaly_indices))
    
    # Appliquer les anomalies uniquement aux indices sélectionnés
    for idx, val in zip(anomaly_indices, anomaly_values):
        data_array[idx] += val
    
    return data_array

# Ajout d'anomalies réalistes - version corrigée
temperature = add_anomalies(temperature, 0.0005, 2)
humidity = add_anomalies(humidity, 0.0005, 5)
co = add_anomalies(co, 0.001, 0.005)
nh3 = add_anomalies(nh3, 0.002, 0.01)  # Plus d'anomalies pour NH3

# Clip final pour assurer la cohérence
temperature = np.clip(temperature, params['temp']['min'], params['temp']['max'])
humidity = np.clip(humidity, params['humidity']['min'], params['humidity']['max'])
co = np.clip(co, 0, params['co']['max'])
nh3 = np.clip(nh3, 0, params['nh3']['max'])

# === CRÉATION DU DATASET ===
df = pd.DataFrame({
    'ts': timestamps,
    'device_id': 'sensor_1',
    'temp': np.round(temperature, 2),
    'humidity': np.round(humidity, 2),
    'co': np.round(co, 6),
    'lpg': np.round(lpg, 6),
    'smoke': np.round(smoke, 6),
    'nh3': np.round(nh3, 6),
    'motion': motion,
    'light': light
})

# === VÉRIFICATIONS ET STATISTIQUES ===
print("\n" + "="*50)
print("📊 STATISTIQUES DU DATASET GÉNÉRÉ")
print("="*50)

print(f"\n🌡️  TEMPÉRATURE:")
print(f"   Plage: {df['temp'].min():.1f}°C - {df['temp'].max():.1f}°C")
print(f"   Moyenne: {df['temp'].mean():.1f}°C")
ideal_temp = df[(df['temp'] >= 20) & (df['temp'] <= 24)]
print(f"   Dans plage idéale (20-24°C): {len(ideal_temp)/len(df)*100:.1f}%")

print(f"\n💧 HUMIDITÉ:")
print(f"   Plage: {df['humidity'].min():.1f}% - {df['humidity'].max():.1f}%")
print(f"   Moyenne: {df['humidity'].mean():.1f}%")
ideal_humidity = df[(df['humidity'] >= 50) & (df['humidity'] <= 60)]
print(f"   Dans plage idéale (50-60%): {len(ideal_humidity)/len(df)*100:.1f}%")

print(f"\n📈 GAZ (moyennes):")
print(f"   CO: {df['co'].mean():.6f}")
print(f"   LPG: {df['lpg'].mean():.6f}")
print(f"   Smoke: {df['smoke'].mean():.6f}")
print(f"   NH3: {df['nh3'].mean():.6f}")

print(f"\n⚡ ACTIVITÉ:")
print(f"   Mouvement: {df['motion'].mean()*100:.1f}% du temps")
print(f"   Lumière allumée: {df['light'].mean()*100:.1f}% du temps")

print(f"\n🕒 PATTERNS TEMPORELS:")
day_motion = df[df['light'] == 1]['motion'].mean()
night_motion = df[df['light'] == 0]['motion'].mean()
print(f"   Mouvement jour: {day_motion*100:.1f}% vs nuit: {night_motion*100:.1f}%")

# Corrélations importantes
corr_temp_humidity = df['temp'].corr(df['humidity'])
corr_light_motion = df['light'].corr(df['motion'])
print(f"\n📊 CORRÉLATIONS:")
print(f"   Température-Humidité: {corr_temp_humidity:.3f}")
print(f"   Lumière-Mouvement: {corr_light_motion:.3f}")

print(f"\n✅ Dataset généré: {df.shape}")
print(f"   Période: {df['ts'].min()} to {df['ts'].max()}")
print(f"   Capteur: {df['device_id'].unique()[0]}")

# Sauvegarde
df.to_csv("simulated_poultry_data_2018_2025_enhanced.csv", index=False)
print(f"\n💾 Fichier sauvegardé: simulated_poultry_data_2018_2025_enhanced.csv")

# Aperçu des données
print(f"\n👀 APERÇU DES DONNÉES:")
print(df.head(10))

Génération de données avicoles réalistes...

📊 STATISTIQUES DU DATASET GÉNÉRÉ

🌡️  TEMPÉRATURE:
   Plage: 18.0°C - 28.0°C
   Moyenne: 21.6°C
   Dans plage idéale (20-24°C): 49.3%

💧 HUMIDITÉ:
   Plage: 40.0% - 70.0%
   Moyenne: 55.0%
   Dans plage idéale (50-60%): 57.5%

📈 GAZ (moyennes):
   CO: 0.005897
   LPG: 0.001012
   Smoke: 0.001002
   NH3: 0.007207

⚡ ACTIVITÉ:
   Mouvement: 36.4% du temps
   Lumière allumée: 69.2% du temps

🕒 PATTERNS TEMPORELS:
   Mouvement jour: 47.8% vs nuit: 10.8%

📊 CORRÉLATIONS:
   Température-Humidité: -0.788
   Lumière-Mouvement: 0.355

✅ Dataset généré: (70105, 10)
   Période: 2018-01-01 00:00:00 to 2025-12-31 00:00:00
   Capteur: sensor_1

💾 Fichier sauvegardé: simulated_poultry_data_2018_2025_enhanced.csv

👀 APERÇU DES DONNÉES:
                   ts device_id   temp  humidity        co       lpg  \
0 2018-01-01 00:00:00  sensor_1  20.68     56.53  0.002264  0.001767   
1 2018-01-01 01:00:00  sensor_1  20.15     61.93  0.000943  0.000569   
2 2018-01